# Extensible PAM Clustering Framework for Type 2 Diabetes Data

This notebook compares PAM clustering results across multiple distance metrics. The statistical workflow is unchanged: each distance metric gets its own silhouette-based optimal K, PAM clustering, evaluation metrics, summary statistics, MDS plot, PCA plot, and exported outputs.


In [ ]:
import sys
# ==========================================================
# Install required packages (run once per new Colab runtime)
# ==========================================================

!pip install -q numpy==1.26.4 scikit-learn-extra==0.3.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 28.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you ha

## 1 Imports


In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.manifold import MDS
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from sklearn_extra.cluster import KMedoids

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2 Configuration

Add new distance metrics here. Downstream analysis reads this configuration automatically.


In [ ]:
DATASET_PATH = "nhanes_1999_2020_t2d_ddrtree_legacy_rank_resid_age_dx_sex_race.csv.xlsx"
RANDOM_STATE = 42
K_MIN = 2
K_MAX = 10

RAW_FEATURES = [
    "HbA1c",
    "BMI",
    "Total_Chol",
    "HDL_C",
    "Triglyceride",
    "ALT",
    "Creatinine",
    "SBP",
    "DBP",
]

# To add most scipy-supported distances, add one new entry here.
# Example:
# "Cosine": {
#     "kind": "scipy",
#     "metric": "cosine",
#     "file_name": "cosine_distance_matrix.csv",
# }
#
# Example for Mahalanobis:
# "Mahalanobis": {
#     "kind": "scipy",
#     "metric": "mahalanobis",
#     "metric_kwargs_factory": lambda values: {
#         "VI": np.linalg.pinv(np.cov(values, rowvar=False))
#     },
#     "file_name": "mahalanobis_distance_matrix.csv",
# }
DISTANCE_METHODS = {
    "Euclidean": {
        "kind": "scipy",
        "metric": "euclidean",
        "file_name": "euclidean_distance_matrix.csv",
    },
    "Manhattan": {
        "kind": "scipy",
        "metric": "cityblock",
        "file_name": "manhattan_distance_matrix.csv",
    },
    "RandomForest": {
        "kind": "random_forest",
        "n_estimators": 500,
        "file_name": "random_forest_distance_matrix.csv",
    },
    "Cosine": {
        "kind": "scipy",
        "metric": "cosine",
        "file_name": "cosine_distance_matrix.csv",
    },
    "Correlation": {
        "kind": "scipy",
        "metric": "correlation",
        "file_name": "correlation_distance_matrix.csv",
    },
    "Canberra": {
        "kind": "scipy",
        "metric": "canberra",
        "file_name": "canberra_distance_matrix.csv",
    },
    "Chebyshev": {
        "kind": "scipy",
        "metric": "chebyshev",
        "file_name": "chebyshev_distance_matrix.csv",
    },
    "BrayCurtis": {
        "kind": "scipy",
        "metric": "braycurtis",
        "file_name": "braycurtis_distance_matrix.csv",
    },
    "SquaredEuclidean": {
        "kind": "scipy",
        "metric": "sqeuclidean",
        "file_name": "squared_euclidean_distance_matrix.csv",
    },
    "Minkowski": {
        "kind": "scipy",
        "metric": "minkowski",
        "file_name": "minkowski_distance_matrix.csv",
    },
}

## 3 Load Dataset


In [ ]:
df = pd.read_excel(DATASET_PATH)

df.head()


In [ ]:
df.columns


## 4 Data Preparation


In [ ]:
# Keep the clinical variables used in the original analysis.
# Rows with missing values are removed so every metric uses the same patients.
X = df[RAW_FEATURES].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X.head()


## 5 Framework Functions


In [ ]:
def build_scipy_distance_matrix(X_scaled, method_config):
    """Build a distance matrix using scipy.spatial.distance.pdist."""
    metric = method_config["metric"]

    metric_kwargs_factory = method_config.get("metric_kwargs_factory")
    if metric_kwargs_factory is None:
        metric_kwargs = method_config.get("metric_kwargs", {})
    else:
        metric_kwargs = metric_kwargs_factory(X_scaled)

    condensed_distances = pdist(
        X_scaled,
        metric=metric,
        **metric_kwargs,
    )

    distance_matrix = squareform(condensed_distances)
    return distance_matrix


def build_random_forest_distance_matrix(X_scaled, method_config, random_state=42):
    """Build the unsupervised Random Forest proximity distance matrix."""
    rng = np.random.default_rng(random_state)
    X_synthetic = np.zeros_like(X_scaled)

    for col in range(X_scaled.shape[1]):
        X_synthetic[:, col] = rng.permutation(X_scaled[:, col])

    X_combined = np.vstack([X_scaled, X_synthetic])
    y_combined = np.array([1] * len(X_scaled) + [0] * len(X_synthetic))

    rf = RandomForestClassifier(

     n_estimators=method_config.get("n_estimators", 1000),

     max_depth=10,

     min_samples_leaf=5,

     min_samples_split=10,

      max_features="sqrt",

     bootstrap=True,

     random_state=random_state,

      n_jobs=-1,
  )
    rf.fit(X_combined, y_combined)

    leaf_matrix = rf.apply(X_scaled)
    print("Leaf matrix shape:", leaf_matrix.shape)

    unique_leaf_counts = [
        len(np.unique(leaf_matrix[:, tree]))
        for tree in range(leaf_matrix.shape[1])
    ]

    print("Average unique leaves per tree:",
          np.mean(unique_leaf_counts))

    print("Minimum unique leaves:",
          np.min(unique_leaf_counts))

    print("Maximum unique leaves:",
          np.max(unique_leaf_counts))
    n_samples = leaf_matrix.shape[0]
    proximity_matrix = np.zeros((n_samples, n_samples))

    for tree in range(leaf_matrix.shape[1]):
        leaves = leaf_matrix[:, tree]
        proximity_matrix += leaves[:, None] == leaves[None, :]

    proximity_matrix = proximity_matrix / leaf_matrix.shape[1]
    print("Mean proximity:", proximity_matrix.mean())
    print("Median proximity:", np.median(proximity_matrix))
    print("Maximum proximity:", proximity_matrix.max())
    print("Minimum proximity:", proximity_matrix.min())

    print("Proximity > 0:",
          np.mean(proximity_matrix > 0))

    print("Proximity > 0.5:",
          np.mean(proximity_matrix > 0.5))
    distance_matrix = 1 - proximity_matrix

    return distance_matrix


def build_distance_matrix(X_scaled, method_name, method_config, random_state=42):
    """Select the correct distance builder from the metric configuration."""
    kind = method_config["kind"]

    if kind == "scipy":
        distance_matrix = build_scipy_distance_matrix(
            X_scaled,
            method_config,
        )
    elif kind == "random_forest":
        distance_matrix = build_random_forest_distance_matrix(
            X_scaled,
            method_config,
            random_state=random_state,
        )
    else:
        raise ValueError(f"Unknown distance method kind for {method_name}: {kind}")

    return distance_matrix


def make_distance_dataframe(distance_matrix, index):
    """Convert a NumPy distance matrix into a labelled DataFrame."""
    distance_df = pd.DataFrame(
        distance_matrix,
        index=index,
        columns=index,
    )

    return distance_df


def generate_distance_matrices(X_scaled, index, distance_methods, random_state=42):
    """Generate every configured distance matrix and store it by method name."""
    distance_results = {}

    for method_name, method_config in distance_methods.items():
        print(f"Generating distance matrix for {method_name}")

        distance_matrix = build_distance_matrix(
            X_scaled,
            method_name,
            method_config,
            random_state=random_state,
        )
        distance_df = make_distance_dataframe(distance_matrix, index)

        distance_results[method_name] = {
            "config": method_config,
            "distance_matrix": distance_matrix,
            "distance_df": distance_df,
        }

    return distance_results

In [ ]:
def run_pam(distance_matrix, n_clusters, random_state=42):
    """Run PAM clustering for one precomputed distance matrix."""
    pam = KMedoids(
        n_clusters=n_clusters,
        metric="precomputed",
        method="pam",
        random_state=random_state,
    )

    labels = pam.fit_predict(distance_matrix)
    return labels


def determine_optimal_k(distance_matrix, k_min=2, k_max=10, random_state=42):
    """Evaluate PAM across K values and choose the K with the best silhouette score."""
    max_possible_k = min(k_max, distance_matrix.shape[0] - 1)
    silhouette_rows = []
    labels_by_k = {}

    for k in range(k_min, max_possible_k + 1):
        labels = run_pam(
            distance_matrix,
            n_clusters=k,
            random_state=random_state,
        )
        score = silhouette_score(
            distance_matrix,
            labels,
            metric="precomputed",
        )

        silhouette_rows.append(
            {
                "K": k,
                "Silhouette Score": score,
            }
        )
        labels_by_k[k] = labels

    silhouette_table = pd.DataFrame(silhouette_rows)
    best_row = silhouette_table.loc[
        silhouette_table["Silhouette Score"].idxmax()
    ]
    best_k = int(best_row["K"])
    best_labels = labels_by_k[best_k]

    return best_k, best_labels, silhouette_table


def compute_evaluation_metrics(distance_matrix, X_scaled, labels):
    """Calculate cluster quality metrics for one clustering result."""
    evaluation = {
        "Optimal K": len(np.unique(labels)),
        "Silhouette Score": silhouette_score(
            distance_matrix,
            labels,
            metric="precomputed",
        ),
        "Davies-Bouldin Index": davies_bouldin_score(X_scaled, labels),
        "Calinski-Harabasz Index": calinski_harabasz_score(X_scaled, labels),
    }

    return evaluation


def compute_summary_statistics(X, labels):
    """Calculate mean, standard deviation, and count by cluster."""
    summary = X.copy()
    summary["Cluster"] = labels

    summary_statistics = summary.groupby("Cluster").agg(["mean", "std", "count"])
    return summary_statistics


def compute_cluster_means(X, labels):
    """Calculate representative mean profiles by cluster."""
    summary = X.copy()
    summary["Cluster"] = labels

    cluster_means = summary.groupby("Cluster").mean()
    return cluster_means


In [ ]:
def plot_silhouette_scores(silhouette_table, method_name):
    """Plot silhouette score for every tested K."""
    fig, ax = plt.subplots(figsize=(7, 4))

    ax.plot(
        silhouette_table["K"],
        silhouette_table["Silhouette Score"],
        marker="o",
    )
    ax.set_title(f"{method_name}: Silhouette Score by K")
    ax.set_xlabel("Number of clusters (K)")
    ax.set_ylabel("Silhouette Score")
    ax.set_xticks(silhouette_table["K"])
    ax.grid(True, alpha=0.3)

    plt.show()
    return fig


def plot_mds(distance_matrix, labels, method_name, random_state=42):
    """Create a two-dimensional MDS plot from a precomputed distance matrix."""
    mds = MDS(
        n_components=2,
        dissimilarity="precomputed",
        random_state=random_state,
    )

    embedding = mds.fit_transform(distance_matrix)

    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
    )
    ax.set_title(f"{method_name}: MDS Plot")
    ax.set_xlabel("MDS Dimension 1")
    ax.set_ylabel("MDS Dimension 2")
    ax.legend(*scatter.legend_elements(), title="Cluster")

    plt.show()
    return embedding, fig


def plot_pca(X_scaled, labels, method_name):
    """Create a two-dimensional PCA plot colored by cluster labels."""
    pca = PCA(n_components=2, random_state=42)
    embedding = pca.fit_transform(X_scaled)

    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=labels,
        cmap="tab10",
        s=35,
    )
    ax.set_title(f"{method_name}: PCA Plot")
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
    ax.legend(*scatter.legend_elements(), title="Cluster")

    plt.show()
    return embedding, fig


In [ ]:
def analyse_distance_metric(
    method_name,
    method_result,
    X,
    X_scaled,
    k_min=2,
    k_max=10,
    random_state=42,
):
    """Run the complete clustering analysis for one distance metric."""
    distance_matrix = method_result["distance_matrix"]

    best_k, labels, silhouette_table = determine_optimal_k(
        distance_matrix,
        k_min=k_min,
        k_max=k_max,
        random_state=random_state,
    )

    evaluation = compute_evaluation_metrics(
        distance_matrix,
        X_scaled,
        labels,
    )
    summary_statistics = compute_summary_statistics(X, labels)
    cluster_means = compute_cluster_means(X, labels)

    silhouette_plot = plot_silhouette_scores(
        silhouette_table,
        method_name,
    )
    mds_embedding, mds_plot = plot_mds(
        distance_matrix,
        labels,
        method_name,
        random_state=random_state,
    )
    pca_embedding, pca_plot = plot_pca(
        X_scaled,
        labels,
        method_name,
    )

    complete_result = {
        **method_result,
        "best_k": best_k,
        "labels": labels,
        "silhouette_table": silhouette_table,
        "evaluation": evaluation,
        "summary_statistics": summary_statistics,
        "cluster_means": cluster_means,
        "mds_embedding": mds_embedding,
        "pca_embedding": pca_embedding,
        "plots": {
            "silhouette": silhouette_plot,
            "mds": mds_plot,
            "pca": pca_plot,
        },
    }

    return complete_result


## 6 Generate Distance Matrices


In [ ]:
results = generate_distance_matrices(
    X_scaled,
    X.index,
    DISTANCE_METHODS,
    random_state=RANDOM_STATE,
)

for method_name, method_result in results.items():
    print(method_name)
    display(method_result["distance_df"].head())


## 7 Run Complete Analysis Pipeline


In [ ]:
for method_name, method_result in results.items():
    print(f"Running complete analysis for {method_name}")

    results[method_name] = analyse_distance_metric(
        method_name,
        method_result,
        X,
        X_scaled,
        k_min=K_MIN,
        k_max=K_MAX,
        random_state=RANDOM_STATE,
    )

    print(f"Best K for {method_name}: {results[method_name]['best_k']}")
    display(results[method_name]["silhouette_table"])


## 8 Cluster Assignments and Sizes


In [ ]:
pam_results = {
    method_name: method_result["labels"]
    for method_name, method_result in results.items()
}

pam_cluster_table = pd.DataFrame(
    pam_results,
    index=X.index,
)

pam_cluster_table.head()


In [ ]:
for method_name in pam_cluster_table.columns:
    print("\n", method_name)
    print(pam_cluster_table[method_name].value_counts().sort_index())


## 9 Cluster Evaluation


In [ ]:
evaluation_table = (
    pd.DataFrame({
        method_name: method_result["evaluation"]
        for method_name, method_result in results.items()
    })
    .T
    .reset_index()
    .rename(columns={"index": "Distance Metric"})
)

evaluation_table


In [ ]:
comparison_results = []

for method1 in results:
    for method2 in results:
        ari_score = adjusted_rand_score(
            results[method1]["labels"],
            results[method2]["labels"],
        )

        comparison_results.append(
            [
                method1,
                method2,
                ari_score,
            ]
        )

ari_table = pd.DataFrame(
    comparison_results,
    columns=[
        "Method 1",
        "Method 2",
        "ARI Score",
    ],
)

ari_table


## 10 Cluster Summary Statistics


In [ ]:
for method_name, method_result in results.items():
    print(f"{method_name}: Summary Statistics")
    display(method_result["summary_statistics"])

    print(f"{method_name}: Representative Mean Cluster Profiles")
    display(method_result["cluster_means"])


In [ ]:
# Backwards-compatible names for older cells or notes that referenced these objects.
euclidean_distance_matrix = results["Euclidean"]["distance_matrix"]
manhattan_distance_matrix = results["Manhattan"]["distance_matrix"]
rf_distance_matrix = results["RandomForest"]["distance_matrix"]

euclidean_distance_df = results["Euclidean"]["distance_df"]
manhattan_distance_df = results["Manhattan"]["distance_df"]
rf_distance_df = results["RandomForest"]["distance_df"]

euclidean_summary = X.copy()
euclidean_summary["Cluster"] = results["Euclidean"]["labels"]
euclidean_cluster_means = results["Euclidean"]["cluster_means"]

manhattan_summary = X.copy()
manhattan_summary["Cluster"] = results["Manhattan"]["labels"]
manhattan_cluster_means = results["Manhattan"]["cluster_means"]

rf_summary = X.copy()
rf_summary["Cluster"] = results["RandomForest"]["labels"]
rf_cluster_means = results["RandomForest"]["cluster_means"]


## 11 Exports


In [ ]:
for method_name, method_result in results.items():
    file_name = method_result["config"].get(
        "file_name",
        f"{method_name}_distance_matrix.csv",
    )
    method_result["distance_df"].to_csv(file_name)

pam_cluster_table.to_csv("PAM_three_distance_results.csv")
evaluation_table.to_csv("PAM_final_evaluation_table.csv", index=False)
ari_table.to_csv("PAM_distance_comparison_ARI.csv", index=False)

for method_name, method_result in results.items():
    method_result["silhouette_table"].to_csv(
        f"{method_name}_silhouette_scores.csv",
        index=False,
    )
    method_result["summary_statistics"].to_csv(
        f"{method_name}_cluster_summary_statistics.csv",
    )
    method_result["cluster_means"].to_csv(
        f"{method_name}_cluster_mean_profiles.csv",
    )

print("All clustering outputs saved.")
